# Demo 3: The Care-Management Patient List

**Project so far.** Demo 1 found the clinics that miss the wait target, and Demo 2 delivered the Quality Committee's fourth-quarter packet. The board approved the diabetes care-management pilot at Bayview and Excelsior, the two clinics in the most deprived neighborhoods.

**This phase's ask.** Dana Okafor, RN, manages the pilot's four care managers, and the pilot starts on February 2, 2026. She needs a list of Bayview and Excelsior patients whose latest A1c is 9% or more, poorly controlled diabetes, with each patient's number of A1c tests, mean, spread, highest, and latest value. The list is rebuilt every night from the lab extract on the secure analytics server, so the build must be fast and fit in memory. Two questions decide the pilot's size: how many patients qualify, and whether four care managers, at 150 patients each, can cover them. The source is the 2025 lab extract: 2,000,000 synthetic results, one row per result.

Run the cells from top to bottom.

In [ ]:
# Installs the course's pandas, pyarrow (for Parquet), and dowhy (for the data generator) in Colab (uv sync already did locally); if Colab asks, restart and rerun from the top
%pip install -q --no-warn-conflicts pandas==3.0.5 pyarrow==25.0.0 dowhy==0.14

This supplied cell downloads `generate_data.py` when they are missing, runs `generate_data.build()` to write the tables into `data/` if they are not there yet (about 30 seconds), then loads the lab results.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

REPO_RAW = "https://raw.githubusercontent.com/christopherseaman/datasci_217/main/08/demo"
if not Path("generate_data.py").exists():
    urlretrieve(f"{REPO_RAW}/generate_data.py", "generate_data.py")
if not all(Path(f"data/{name}.parquet").exists() for name in ["clinics", "visits", "labs"]):
    import generate_data
    generate_data.build()

labs = pd.read_parquet("data/labs.parquet")
print(labs.shape)

Expect `(2000000, 6)`.

## Core walkthrough

### 1. What is in the lab extract?

In [ ]:
display(labs.head())
display(labs["test_name"].value_counts())
print(f"Patients: {labs['patient_id'].nunique():,}")
print(f"No result (cancelled samples): {labs['result'].isna().sum():,}")

Expect five results with patient, clinic, collection time, test, result, and unit, then four tests from Glucose (887,885 results) and Creatinine (887,656), drawn together as one panel, down to LDL (105,134), then 188,757 patients and 10,890 results with no value. One row per result, so one blood draw can fill several rows.

### 2. Can the extreme results be trusted?

Before building features from this extract, look at the most extreme results at each site; a bad code there would distort every mean and maximum. `apply` hands each clinic's rows to a function, here `nlargest`, which keeps each clinic's two highest glucose results as whole rows.

In [ ]:
glucose = labs[labs["test_name"] == "Glucose"]
top2_raw = glucose.groupby("clinic").apply(lambda g: g.nlargest(2, "result"), include_groups=False)
display(top2_raw[["patient_id", "collected_at", "result"]])

Expect 16 rows, two per clinic, indexed by clinic and then each result's original row number. The other clinics' highest results are 442 to 740 mg/dL, emergencies worth a call, but Excelsior's two are `9999.0`: no blood holds that much glucose. At five clinics both belong to one patient, such as P118809 at Bayview, 3 months apart. 9999 is the code one Excelsior analyzer writes for a failed run, so it is not a result.

Mark the error codes as missing (Lecture 05) for the whole extract, then repeat the review. The nightly job will run this check every night, so also try the `apply`-free version: sort once, then keep the first two rows of each clinic.

In [ ]:
print("Error codes:", (labs["result"] == 9999).sum())
labs.loc[labs["result"] == 9999, "result"] = np.nan

glucose = labs[labs["test_name"] == "Glucose"]
top2 = glucose.groupby("clinic").apply(lambda g: g.nlargest(2, "result"), include_groups=False)
top2_sorted = glucose.sort_values("result", ascending=False).groupby("clinic").head(2)
display(top2_sorted[["clinic", "patient_id", "result"]].sort_values("clinic"))
print("Same rows both ways?", sorted(top2.reset_index()["level_1"]) == sorted(top2_sorted.index))

Expect `Error codes: 23`, then 16 rows with a flat index; Excelsior's highest is now a real result, 538.0 mg/dL. Then `True`: both ways keep the same rows, so the nightly check can use the faster sort-and-`head` version.

Plot the two reviews side by side, one dot per result.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
sns.scatterplot(data=top2_raw.reset_index(), x="result", y="clinic", s=60, ax=axes[0])
axes[0].set(title="Before: two highest glucose per clinic", xlabel="Glucose (mg/dL)", ylabel="")
sns.scatterplot(data=top2.reset_index(), x="result", y="clinic", s=60, ax=axes[1])
axes[1].set(title="After marking 9999 as missing", xlabel="Glucose (mg/dL)")
plt.show()

Expect the left panel's dots bunched under 750 at the left edge, except Excelsior's dot alone at 9999 (its two results are identical, so they overlap). On the right the x-axis runs from about 440 to 750, and every clinic's two dots sit between 442 and 740 mg/dL.

### 3. How many patients would the pilot take on?

The pilot enrolls on the most recent A1c, not the average of old ones. Sort by collection time, then keep each patient's last row.

In [ ]:
a1c = labs[labs["test_name"] == "A1c"].dropna(subset=["result"])
latest = a1c.sort_values("collected_at").groupby("patient_id").tail(1)
display(latest.head())
print(f"A1c results: {len(a1c):,}  patients: {a1c['patient_id'].nunique():,}  latest rows: {len(latest):,}")
print(f"Latest A1c of 9% or more: {(latest['result'] >= 9).mean():.1%}")

Expect five latest rows (patients whose only A1c was drawn on January 2), then 118,645 A1c results from 90,178 patients and exactly one latest row per patient. 2.0% of patients have a latest A1c of 9% or more, poorly controlled diabetes; most of the others had one yearly screening test.

The red lines mark the diabetes cutoff (6.5%) and poor control (9%); `ax.axvline(x)` draws a vertical line at `x`.

In [ ]:
ax = latest["result"].plot.hist(bins=np.arange(3.95, 15.1, 0.1), figsize=(8, 3.5))
for cutoff in [6.5, 9]:
    ax.axvline(cutoff, color="red", linestyle="--")
ax.set(title="Latest A1c per patient, 2025", xlabel="A1c (%)", ylabel="Patients")
plt.show()

Expect a tall peak near 5.3%, below the diabetes cutoff at 6.5%, and a second, lower hump near 7.5% from patients with diabetes. The thin tail past the 9% line is the 2.0%.

Now the pilot's own count: patients whose latest A1c is 9% or more at the two pilot clinics.

In [ ]:
pilot = latest[latest["clinic"].isin(["Bayview", "Excelsior"]) & (latest["result"] >= 9)]
display(pilot["clinic"].value_counts())
print(f"Pilot patients: {len(pilot):,}, about {len(pilot) / 4:.0f} per care manager")

Expect 378 patients at Bayview and 290 at Excelsior, 668 in all, about 167 per care manager.

**What we tell Ms. Okafor:** 668 patients qualify. At her target caseload of 150, four care managers cover 600, so either the pilot adds a fifth care manager or it starts with the 600 highest latest A1c values.

### 4. The draft feature build splits the table four times

The list needs each patient's test count, mean, spread, highest, and latest A1c every night. A colleague's draft computes each feature with its own `groupby()` call. Before changing a nightly job, check that one `.agg()` call gives the same numbers, then time both.

In [ ]:
def four_calls(df):
    """Count, mean, SD, and max of A1c per patient, one groupby() call each."""
    return [df.groupby("patient_id")["result"].agg(f) for f in ["count", "mean", "std", "max"]]

def one_agg(df):
    """The same four summaries from one groupby() and one .agg() call."""
    return df.groupby("patient_id")["result"].agg(["count", "mean", "std", "max"])

together = one_agg(a1c)
print(all(part.equals(together[name]) for part, name in zip(four_calls(a1c), together.columns)))

Expect `True`.

In [ ]:
%timeit four_calls(a1c)
%timeit one_agg(a1c)

Expect two lines such as `175 ms ± 1.52 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)` and `50.8 ms ± 356 μs per loop (...)`. The one `.agg()` is about 3.5 times faster, because every `groupby()` call splits the 118,645 rows again. Timings depend on the computer, so compare the ratio, not the milliseconds. The rewrite gives the same numbers in under a third of the time.

### 5. The draft's spread column uses a lambda

The draft computes each patient's standard deviation with `agg(lambda s: s.std())`. Check that the built-in `"std"` gives the same values, then time both.

In [ ]:
by_patient = a1c.groupby("patient_id")["result"]
print("Largest difference:", (by_patient.agg("std") - by_patient.agg(lambda s: s.std())).abs().max())

Expect a number near `1e-15`: the same values, apart from rounding in the last decimal place.

The lambda runs as Python once per patient, so one run takes seconds; `-n 1 -r 3` asks `%timeit` for three single runs instead of its usual dozens.

In [ ]:
%timeit by_patient.agg("std")
%timeit -n 1 -r 3 by_patient.agg(lambda s: s.std())

Expect about 3 ms against about 4 s: the lambda is over 1,000 times slower, because it runs once for each of 90,178 patients while `"std"` is one compiled loop over all of them. The build uses `"std"`.

### 6. The server team asks how much memory the job needs

The nightly job loads the whole lab table. `memory_usage(deep=True)` shows which columns take the space, and `astype("category")` stores a repeated text key as a few labels plus a small code per row.

In [ ]:
before = labs.memory_usage(deep=True)
compact = labs.copy()
for column in ["clinic", "test_name", "unit"]:
    compact[column] = compact[column].astype("category")
after = compact.memory_usage(deep=True)
memory = pd.DataFrame({"text": before / 1e6, "category": after / 1e6}).drop(index="Index")
ax = memory.plot.barh(figsize=(7, 3.5))
ax.set(title="Memory per column, 2,000,000 lab results", xlabel="MB")
plt.show()
print(f"Total: {before.sum() / 1e6:.0f} MB as text, {after.sum() / 1e6:.0f} MB with category keys")

same = compact.groupby("test_name", observed=True)["result"].mean().equals(labs.groupby("test_name")["result"].mean())
print("Same test means?", same)

Expect the `clinic`, `test_name`, and `unit` bars to shrink from 25 to 32 MB each to 2.0 MB each (one small code per row plus a few labels), while `patient_id` and the number columns stay the same. The total falls from 150 MB to 68 MB, and the test means are unchanged (`True`). `patient_id` stays text: with 188,757 distinct values, it is not a repeated key.

**What we tell the server team:** the job holds about 150 MB as loaded and 68 MB once it converts the three repeated text keys to `category`.

## Independent practice

Run the core walkthrough first. Each item is a request that came back after the list went out.

### 7. The care managers ask who is already improving

A patient whose A1c is already falling may need less outreach. Among all patients with a latest A1c of 6.5% or more, sort inside each patient with `apply`, keep the last two results, and compare them.

In [ ]:
dm = a1c[a1c["patient_id"].isin(latest.loc[latest["result"] >= 6.5, "patient_id"])]
last_two = dm.groupby("patient_id").apply(
    lambda g: g.sort_values("collected_at").tail(2), include_groups=False,
)
display(last_two[["collected_at", "result"]].head(6))

change = last_two.groupby("patient_id")["result"].agg(["first", "last", "count"])
change = change[change["count"] == 2]
change["change"] = change["last"] - change["first"]
print(f"Patients with two results: {len(change):,}")
print(f"Improved by 0.5 points or more: {(change['change'] <= -0.5).mean():.1%}")

Expect a two-level index (patient, then original row number) with up to two rows per patient, oldest first; P000028 went from 7.3 to 7.1, P000039 from 8.7 to 8.4, and P000002 and P000013 have only one result. 15,716 patients with a latest A1c of 6.5% or more have two results, and 8.7% of them improved by half a point or more between their last two tests.

### 8. A reviewer wants each result beside the patient's own mean

A grouped `transform` puts each patient's mean on every result row. Compare the built-in `transform("mean")` with a `lambda` version, for values and for speed.

In [ ]:
centered_fast = a1c["result"] - by_patient.transform("mean")
centered_slow = by_patient.transform(lambda s: s - s.mean())
print("Largest difference:", (centered_fast - centered_slow).abs().max())
print("Same index as a1c?", centered_fast.index.equals(a1c.index))

Expect a difference near `4e-15` and `True`.

The lambda takes more than 10 seconds on all 90,178 patients, so `-r 1` times one run.

In [ ]:
%timeit a1c["result"] - by_patient.transform("mean")
%timeit -n 1 -r 1 by_patient.transform(lambda s: s - s.mean())

Expect about 3 ms against about 15 s: over 4,000 times slower.

### 9. Dr. Alvarez asks how the pilot clinics compare with the rest

A function that returns a `Series` gives one summary row per clinic through `apply`.

In [ ]:
def a1c_summary(group):
    """Summarize one clinic's A1c results as a Series."""
    return pd.Series({
        "results": len(group),
        "median": group["result"].median(),
        "share_9_or_more": (group["result"] >= 9).mean(),
    })

clinic_summary = a1c.groupby("clinic").apply(a1c_summary, include_groups=False)
display(clinic_summary.round(3))
print("Same medians as agg('median')?", clinic_summary["median"].equals(a1c.groupby("clinic")["result"].median()))

Expect one row per clinic (the function returns one `Series` per group), medians from 5.6% (Marina) to 6.7% (Bayview), 2.8% to 5.1% of results at 9% or more, and `True`. Bayview and Excelsior, in the most deprived neighborhoods, have the highest medians (6.7% and 6.5%) and, with Ridgecrest, the highest shares; Marina, in the least deprived, is lowest on both, which supports the board's choice of pilot clinics. `results` shows `.0` because a `Series` holding a share stores every value as a float.

### 10. Do category keys also speed up the build?

In [ ]:
%timeit labs.groupby("test_name")["result"].mean()
%timeit compact.groupby("test_name", observed=True)["result"].mean()

Expect the category version about 2 times faster: pandas groups small integer codes instead of comparing text.